# 🧬 MedMNIST2D — 3-Stage Pattern Recognition Pipeline

> **8 MedMNIST2D datasets** · Baseline Manifold Learning · Custom SNN Hybrid · Massive Topology & Noise Immunity

This notebook orchestrates a fully modular, memory-efficient pipeline implemented in the `modules/` package:

| Stage | Description |
|-------|-------------|
| **1** | UMAP dimensionality reduction → GNB, LDA, QDA, GMM (soft-clustering) |
| **2** | SOM-ART clustering → Tanh→GeLU Gaussian extraction → RBF layer → Adam+SGLD |
| **3** | MAIS dilution → entropy selection → 29-method cascade augmentation → ADASYN/Tomek → SLERP expansion → Randomized SVD → HDBSCAN coreset → Barnes-Hut t-SNE → 30-model BMA ensemble |


In [2]:
import sys, os, gc, warnings
warnings.filterwarnings('ignore')

# Ensure the project root is on the path
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import numpy as np
import torch

print(f"NumPy  : {np.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"Device : {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")
print(f"Python : {sys.version}")


ModuleNotFoundError: No module named 'numpy'

---
## 📦 Module Overview

In [ ]:
# Quick sanity check that all modules import correctly
from modules.data_loader import MEDMNIST2D_DATASETS, load_medmnist_flat
from modules.metrics    import record_result, build_comparison_table, display_final_table
from modules.stage1     import run_stage1
from modules.stage2     import run_stage2
from modules.stage3     import run_stage3

print("All datasets:", MEDMNIST2D_DATASETS)
print("\n✓ All modules imported successfully.")


---
## Stage 1 — Baseline Manifold Learning & Probabilistic Classifiers

### Mathematical Intuition

**UMAP** (Uniform Manifold Approximation and Projection) models the high-dimensional data as a fuzzy simplicial complex on a Riemannian manifold, then optimises a low-dimensional layout that preserves the topological structure via cross-entropy minimisation between the two fuzzy set representations.

**Classifiers:**
- **GNB:** Assumes feature independence; models $P(x_j | C_k) = \mathcal{N}(\mu_{jk}, \sigma_{jk}^2)$.
- **LDA:** Projects onto the subspace that maximises $\frac{|S_B|}{|S_W|}$ (between/within scatter ratio).
- **QDA:** Like LDA but allows each class its own covariance $\Sigma_k$.
- **GMM:** Per-class Gaussian Mixtures → soft membership $P(C_k | x) \propto \pi_k \sum_m w_{km} \mathcal{N}(x; \mu_{km}, \Sigma_{km})$.


In [ ]:
# Run Stage 1 on ALL datasets
os.makedirs('./data', exist_ok=True)
run_stage1(datasets=None, umap_dim=50, root='./data')


---
## Stage 2 — Custom Spiking Neural Network / Hybrid Architecture

### Mathematical Intuition

**SOM-ART Clustering:**  
The Self-Organizing Map discovers a topological mapping from input space to a 2-D grid via competitive Hebbian learning: $w_i \leftarrow w_i + \eta \cdot h(i, i^*) \cdot (x - w_i)$. Adaptive Resonance Theory then enforces a vigilance parameter $\rho$; if a node's average cosine similarity to its members falls below $\rho$, the cluster is split (bisecting K-Means), yielding finer granularity.

**Tanh → GeLU with Negative-Axis Gaussian Extraction:**  
$\text{GeLU}(x) = x \cdot \Phi(x)$. On the negative axis (post-Tanh), this curve approximates a reflected Gaussian bump. We model this explicitly:

$$g(x) = \exp\left(-\frac{(x - \mu)^2}{2\sigma^2}\right)$$

The gate $g$ is applied **only** on the negative side, allowing the network to extract and amplify the Gaussian structure inherent in GeLU's negative regime.

**RBF Layer:**  
$\phi_k(x) = \exp(-\beta_k \|x - c_k\|^2)$. Centres $c_k$ are determined via K-Means or Fuzzy C-Means. Learnable $\beta_k$ controls each centre's receptive field width.

**Adam + SGLD:**  
After each Adam parameter update, Langevin noise $\epsilon \sim \mathcal{N}(0, 2\eta T)$ is injected, turning the optimiser into an approximate MCMC sampler of the posterior—balancing exploitation (Adam) with exploration (SGLD).


In [ ]:
# Run Stage 2 on ALL datasets
run_stage2(
    datasets=None,
    epochs=20,
    batch_size=256,
    hidden1=128,
    hidden2=64,
    n_rbf=32,
    rbf_method='fcm',   # try 'fcm' for Fuzzy C-Means centres
    lr=1e-3,
    sgld_temp=1e-4,
    root='./data'
)


---
## Stage 3 — Massive Topology & Noise Immunity Pipeline

### Mathematical Intuition

| Step | Method | Key Equation / Idea |
|------|--------|---------------------|
| 1 | **MAIS + ESS** | $\text{ESS} = \frac{1}{\sum w_i^2}$; subsets below threshold are discarded. |
| 2 | **Entropy Selection** | $H(x) = -\sum p_j \log p_j$; lowest-$H$ samples are most confident. |
| 3 | **Cascade Augmentation** | 29 augmentations, each $P(\text{trigger})=0.5$ (stochastic cascade). |
| 4 | **ADASYN + Tomek** | Synthetic minority oversampling + borderline pair removal. |
| 5 | **SLERP Expansion** | $\text{slerp}(v_0, v_1; t) = \frac{\sin((1-t)\omega)}{\sin\omega} v_0 + \frac{\sin(t\omega)}{\sin\omega} v_1$ in PCA space. |
| 6 | **Randomized SVD** | Preserve 95% of Frobenius norm energy: $\|A\|_F^2 \approx \sum_{i=1}^k \sigma_i^2$. |
| 7 | **Coreset** | Score = $\sqrt{p_{\text{HDBSCAN}} \cdot (1 - \hat{H})}$; top 10%. |
| 8 | **Barnes-Hut t-SNE** | $O(N \log N)$ approximation via space-partitioning trees. |
| 9 | **BMA Ensemble** | $P(y|x) = \sum_k w_k P_k(y|x)$, $w_k \propto \exp(\ell_k)$ (validation log-lik). |

### Ensemble Categories (30 models)

1. **KNN:** Standard, Fuzzy, FCM-KNN Hybrid  
2. **Tree:** DT, Bagging, RF, GBM, XGBoost, CatBoost  
3. **Linear:** Perceptron, LogReg-Sigmoid(OvR), LogReg-Softmax  
4. **SGD-SVM + Nyström:** Linear, RBF, Poly2, Poly3, Sigmoid, GRPF, t-Student, Inv-Multiquadric  
5. **Raw SGD:** Huber, Mod-Huber, Sq-Hinge, Perceptron, LogLoss  

Custom Nyström kernels (GRPF, t-Student, Inverse Multiquadric) extend `BaseEstimator, TransformerMixin`.


In [ ]:
# Run Stage 3 on ALL datasets
run_stage3(datasets=None, root='./data')


---
## 📊 Final Comparative Table — All Stages × All Datasets × All Models


In [ ]:
# Build and display the massive comparative table
final_df = display_final_table()

# Save to CSV for downstream use
final_df.to_csv('medmnist_all_results.csv', index=False)
print(f"\nSaved {len(final_df)} rows to medmnist_all_results.csv")


In [ ]:
# Summary statistics per stage
import pandas as pd
summary = final_df.groupby('stage')[['accuracy','f1','auc']].agg(['mean','std','max'])
print(summary.to_string())


---
## ✅ Pipeline Complete

All 3 stages have been evaluated across 8 MedMNIST2D datasets.  
Results are saved to `medmnist_all_results.csv`.

### Quick Reference — Module Structure

```
modules/
├── __init__.py
├── data_loader.py     # Loading & batching for all datasets
├── metrics.py         # Unified evaluation, visualisation, comparison table
├── stage1.py          # UMAP + GNB/LDA/QDA/GMM
├── stage2.py          # SOM-ART → TanhGeLU → RBF → Adam+SGLD
├── stage3_utils.py    # MAIS, 29 augmentations, SLERP, SVD, coreset, Nyström
└── stage3.py          # Full 9-step pipeline + 30-model BMA ensemble
```
